# Day 19 — Sales Anomaly Detection

Flag unusual daily sales observations for review. An anomaly is not proof of fraud, data error, or stockout.

Method: store-wise rolling median and median absolute deviation (MAD), based on prior observations so the current value is not used in its own baseline.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_DIR.mkdir(parents=True, exist_ok=True)

input_path = DATA_DIR / "store_daily_sales.csv"
if not input_path.exists():
    raise FileNotFoundError(f"Missing {input_path}. Run data preparation first.")
sales = pd.read_csv(input_path, parse_dates=["Date"])
required = ["Store","Date","Sales"]
missing = [c for c in required if c not in sales.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")
sales = sales.sort_values(["Store","Date"]).reset_index(drop=True)
print("Rows:", len(sales), "| Stores:", sales["Store"].nunique())
display(sales.head())


Rows: 844392 | Stores: 1115


,Store,Date,Sales
0,1,2013-01-02,5530
1,1,2013-01-03,4327
2,1,2013-01-04,4486
3,1,2013-01-05,4997
4,1,2013-01-07,7176


## Robust anomaly features

In [8]:
WINDOW = 28
MIN_PERIODS = 14
MAD_MULTIPLIER = 3.5

def add_robust_anomaly_features(group):
    group = group.sort_values("Date").copy()

    prior = group["Sales"].shift(1)
    baseline = prior.rolling(
        WINDOW, min_periods=MIN_PERIODS
    ).median()

    deviation = (prior - baseline).abs()
    mad = deviation.rolling(
        WINDOW, min_periods=MIN_PERIODS
    ).median()

    scale = 1.4826 * mad

    group["Rolling_Baseline"] = baseline
    group["Robust_Scale"] = scale

    group["Robust_Anomaly_Score"] = (
        (group["Sales"] - baseline)
        / scale.replace(0, np.nan)
    )

    group["Is_Anomaly"] = (
        group["Robust_Anomaly_Score"].abs() > MAD_MULTIPLIER
    )

    return group

# Process each store independently without groupby.apply()
parts = []

for store_id, group in sales.groupby("Store", sort=False):
    parts.append(add_robust_anomaly_features(group))

anomalies = pd.concat(parts, ignore_index=True)

display(
    anomalies.loc[
        anomalies["Is_Anomaly"],
        [
            "Store",
            "Date",
            "Sales",
            "Rolling_Baseline",
            "Robust_Anomaly_Score",
        ],
    ].head(20)
)

,Store,Date,Sales,Rolling_Baseline,Robust_Anomaly_Score
122,1,2013-05-31,7893,4925.5,3.572604
297,1,2013-12-23,9528,5655.5,3.627730
390,1,2014-04-14,6803,4029.5,3.891212
942,2,2013-07-15,8528,4727.5,4.319127
1073,2,2013-12-16,10479,4987.0,4.535419
1074,2,2013-12-17,10374,5010.5,3.519953
1136,2,2014-03-03,7332,4362.0,3.548694
1160,2,2014-03-31,9908,4404.5,5.127155
1233,2,2014-06-30,10682,4706.5,5.824306
1463,2,2015-03-30,8733,4309.5,3.677793


In [9]:
summary = anomalies.groupby("Store").agg(
    Observations=("Sales", "size"),
    Anomaly_Count=("Is_Anomaly", "sum")
).reset_index()

summary["Anomaly_Rate_Pct"] = (
    100 * summary["Anomaly_Count"] / summary["Observations"]
)

anomalies.to_csv(
    DATA_DIR / "day19_sales_anomalies.csv",
    index=False
)

summary.to_csv(
    DATA_DIR / "day19_anomaly_summary.csv",
    index=False
)

print("Anomalies flagged:", int(anomalies["Is_Anomaly"].sum()))
display(summary.sort_values("Anomaly_Count", ascending=False).head(10))

Anomalies flagged: 19604


,Store,Observations,Anomaly_Count,Anomaly_Rate_Pct
273,274,932,137,14.699571
675,676,918,68,7.407407
84,85,942,65,6.900212
261,262,942,65,6.900212
692,693,781,61,7.810499
708,709,784,60,7.653061
1028,1029,780,56,7.179487
947,948,928,55,5.926724
452,453,887,50,5.636979
252,253,779,49,6.290116


## Conclusion
- Built a store-wise robust anomaly detector using rolling median and MAD.
- Saved detailed anomaly flags and store-level summaries.
- Anomaly flags require investigation; promotions, holidays, closures, and data-quality issues may explain unusual values.

## Assumptions and Limitations

- **Historical window:** The previous 28 observations are used to estimate normal sales behaviour.
- **Minimum history:** At least 14 prior observations are required for the rolling baseline.
- **Detection threshold:** An absolute robust anomaly score greater than 3.5 is flagged as unusual.
- **Store-specific baseline:** Each store is compared with its own historical sales.
- **Interpretation:** An anomaly is a signal for investigation, not proof of fraud, error, or stockout.
- **Data limitation:** Rossmann Sales represents revenue, not physical product units.